# Telegram duplicate video cleanup

Run the steps **in order**, one at a time. To run a step, tap it, then tap the ▶️ play button on its left.

Nothing gets deleted until **Step 5**, and only if you type `DELETE` there.

## Step 1 — Install (takes ~20 seconds)

In [ ]:
!pip install -q telethon
print("Done ✅")

## Step 2 — Log in

When asked, type:
1. Your **api_id**
2. Your **api_hash**
3. Your **phone number** (with country code, like +1...)
4. The **code Telegram sends to your Telegram app**
5. Your **two-step password**, if you have one

In [ ]:
from telethon import TelegramClient
from telethon.sessions import StringSession

api_id = int(input("api_id: ").strip())
api_hash = input("api_hash: ").strip()

# Session lives only in memory: when you close this page, you're logged out of it.
client = TelegramClient(StringSession(), api_id, api_hash, flood_sleep_threshold=120)
await client.start()
me = await client.get_me()
print(f"Logged in as {me.first_name} ✅")

## Step 3 — See your groups

In [ ]:
groups = []
async for d in client.iter_dialogs():
    if d.is_group or d.is_channel:
        groups.append(d)

for i, d in enumerate(groups, 1):
    print(f"{i:>3}. {d.name}")

## Step 4 — Find duplicates (nothing is deleted)

Type the numbers of the groups to check, from the list above, separated by commas (like `1, 4, 7`).

Two videos count as duplicates when they have **the same file size and the same length**. The **oldest** copy is kept; newer copies are marked for deletion.

In [ ]:
from collections import defaultdict
from telethon.tl.types import InputMessagesFilterVideo, DocumentAttributeVideo

picked = input("Group numbers (e.g. 1, 4, 7): ")
chosen = [groups[int(n) - 1] for n in picked.replace(" ", "").split(",") if n]

def video_key(msg):
    doc = msg.document
    if not doc:
        return None
    duration = next((round(a.duration) for a in doc.attributes
                     if isinstance(a, DocumentAttributeVideo)), None)
    return (doc.size, duration)

def link(group, msg_id):
    ent = group.entity
    if getattr(ent, "username", None):
        return f"https://t.me/{ent.username}/{msg_id}"
    return f"https://t.me/c/{ent.id}/{msg_id}"

to_delete = {}  # group -> list of message ids
for g in chosen:
    print(f"\nScanning: {g.name} ...")
    seen = defaultdict(list)
    count = 0
    # reverse=True goes oldest -> newest, so the first copy found is the one kept
    async for msg in client.iter_messages(g.entity, filter=InputMessagesFilterVideo, reverse=True):
        key = video_key(msg)
        if key:
            seen[key].append(msg.id)
            count += 1
    dupes = {k: ids for k, ids in seen.items() if len(ids) > 1}
    extra = [i for ids in dupes.values() for i in ids[1:]]
    to_delete[g] = extra
    print(f"  {count} videos, {len(extra)} duplicate copies to remove")
    for ids in list(dupes.values())[:3]:
        print(f"  e.g. keep {link(g, ids[0])}  remove {link(g, ids[1])}")

total = sum(len(v) for v in to_delete.values())
print(f"\nTOTAL duplicates found: {total}")
print("Tap a few 'e.g.' links above to double-check they really are the same video.")

## Step 5 — Delete duplicates ⚠️ permanent

Only run this once you're happy with Step 4. Type `DELETE` (capital letters) to confirm.

In [ ]:
if input("Type DELETE to permanently remove the duplicates: ").strip() != "DELETE":
    print("Cancelled — nothing deleted.")
else:
    for g, ids in to_delete.items():
        done = 0
        for start in range(0, len(ids), 100):  # Telegram allows 100 per request
            batch = ids[start:start + 100]
            await client.delete_messages(g.entity, batch)
            done += len(batch)
        print(f"{g.name}: deleted {done}")
    print("Finished ✅")

## Step 6 — Log out (recommended)

In [ ]:
await client.log_out()
print("Logged out ✅ — you can close this page.")